In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_small_q4/checkpoints/post_cell_1_try_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# Filter lineitem where commit date < receipt date on GPU
lsel = lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE
# Keep only the key for the join
flineitem = lineitem[lsel][["L_ORDERKEY"]]

# Apply date filters using string literals to avoid CPU Timestamp creation
osel = (orders.O_ORDERDATE < "1993-11-01") & (orders.O_ORDERDATE >= "1993-08-01")
# Keep only the columns needed downstream
forders = orders[osel][["O_ORDERKEY", "O_ORDERPRIORITY"]]

# Perform an inner join on the GPU rather than isin-based filtering
jn = forders.merge(flineitem, left_on="O_ORDERKEY", right_on="L_ORDERKEY")

# Group and count entirely on the GPU
total = jn.groupby("O_ORDERPRIORITY", as_index=False).O_ORDERKEY.count()